# AutoDL v6 overlap40：从头训练 Gated A（seed1337）

使用 ImageNet ResNet50 初始化，训练 Gated Detail Fusion A；physical batch4、boundary weight 0、80 epochs，仅用 Val 选模，不访问 Test。

## 1. 服务器路径

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/root/autodl-tmp/datasets/dataset_v6_random811_overlap40')
OUTPUT_ROOT = Path('/root/autodl-tmp/outputs')
REPO_DIR = Path('/root/autodl-tmp/projects/lunar-linear')
CONFIG_FILE = 'v6_overlap40_gated_no_boundary_batch4_seed1337.json'

## 2. 环境、代码和输入核验

In [ ]:
import hashlib, importlib.metadata, importlib.util, json, shutil, subprocess, sys

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
required = [('rasterio', 'rasterio'), ('tqdm', 'tqdm')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
import numpy as np
import torch
assert torch.cuda.is_available(), '服务器未识别 GPU'
assert (REPO_DIR / '.git').is_dir(), f'仓库不存在: {REPO_DIR}'
PROJECT_DIR = REPO_DIR / 'LTL-Net'
config = json.loads((PROJECT_DIR / 'configs' / CONFIG_FILE).read_text(encoding='utf-8'))
assert config['module'] == 'gated_boundary'
assert config['seed'] == 1337 and config['epochs'] == 80
assert config['batch_size'] == 4 and config['accum_steps'] == 1
assert config['boundary_weight'] == 0.0
assert config['encoder_weights'] == 'imagenet'
assert config['automatic_test_evaluation'] is False
assert DATA_ROOT.is_dir(), f'数据目录不存在: {DATA_ROOT}'
for split, expected in config['expected_tiles'].items():
    images = sorted([*(DATA_ROOT / split / 'image').glob('*.tif'), *(DATA_ROOT / split / 'image').glob('*.tiff')])
    masks = sorted([*(DATA_ROOT / split / 'mask').glob('*.tif'), *(DATA_ROOT / split / 'mask').glob('*.tiff')])
    assert len(images) == len(masks) == expected, (split, len(images), len(masks))
    assert {p.stem for p in images} == {p.stem for p in masks}
for name, expected_hash in config['expected_metadata_sha256'].items():
    actual_hash = hashlib.sha256((DATA_ROOT / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, (name, actual_hash, expected_hash)
stats = json.loads((DATA_ROOT / 'normalization_stats.json').read_text(encoding='utf-8'))
assert np.allclose(stats['mean'], config['expected_mean'], rtol=0, atol=1e-12)
assert np.allclose(stats['std'], config['expected_std'], rtol=0, atol=1e-12)
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('GPU:', torch.cuda.get_device_name(0))
print('Commit:', commit)
print('Data:', DATA_ROOT)
print(json.dumps(config, ensure_ascii=False, indent=2))

## 3. batch4、512×512 训练冒烟

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
from models.module_models import build_module_model
from train_module_experiment import ExperimentLoss, model_outputs
smoke_model = build_module_model('gated_boundary', encoder_weights=None).cuda().train()
criterion = ExperimentLoss('gated_boundary', boundary_weight=0.0).cuda()
optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=config['learning_rate'])
x = torch.randn(4, 5, 512, 512, device='cuda')
labels = torch.randint(0, 5, (4, 512, 512), device='cuda')
with torch.amp.autocast('cuda'):
    logits, boundary_logits = model_outputs(smoke_model, x, True)
    loss, parts = criterion(logits, labels, boundary_logits)
loss.backward(); optimizer.step()
assert logits.shape == (4, 5, 512, 512) and torch.isfinite(loss)
print('Smoke loss:', float(loss), parts)
del smoke_model, criterion, optimizer, x, labels, logits, boundary_logits, loss
torch.cuda.empty_cache()

## 4. 正式训练 A1337

In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
assert not result_dir.exists(), f'结果目录已存在，为防止覆盖已停止: {result_dir}'
command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'train_module_experiment.py'),
    '--module', config['module'], '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT), '--run-name', config['run_name'],
    '--seed', str(config['seed']), '--epochs', str(config['epochs']),
    '--batch-size', str(config['batch_size']), '--accum-steps', str(config['accum_steps']),
    '--num-workers', str(config['num_workers']), '--learning-rate', str(config['learning_rate']),
    '--boundary-weight', str(config['boundary_weight']), '--encoder-weights', config['encoder_weights'],
]
print(' '.join(command))
subprocess.check_call(command, cwd=PROJECT_DIR)

## 5. 查看并打包结果

In [ ]:
result = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
assert result['test_evaluated'] is False
assert result['seed'] == 1337 and result['model'] == 'gated_boundary'
print(json.dumps(result, ensure_ascii=False, indent=2))
archive_path = shutil.make_archive(str(OUTPUT_ROOT / result_dir.name), 'zip', root_dir=result_dir)
print('结果目录:', result_dir)
print('压缩包:', archive_path)